# 02 · NASR runways and airport metadata
Source: the FAA NASR 28-day subscription APT CSVs in `data/raw/nasr/`.

Outputs:
- `data/interim/runway_ends.parquet`: one row per usable runway end (paved, ≥ 5,000 ft, no helipads), with its **true** heading and ILS flag.
- `data/interim/airports.parquet`: FAA, IATA and ICAO codes, lat/lon and IANA timezone for every NASR airport. BTS times are local, so 04_flights needs these timezones.

Limitation: NASR is a snapshot of the current layout (see `EFF_DATE`), not historical.

In [1]:
%run ../config.ipynb

ROOT = /Users/dylannegvesky/Documents/FSU Documents/Fall 2026/ProjDS/PDS_Final
study airports: ['EWR'] | BTS 2023-01 → latest | train ≤ 2025-12


In [2]:
from timezonefinder import TimezoneFinder

read = lambda name: pd.read_csv(NASR_DIR / name, encoding="latin1", low_memory=False)
base, rwy, end = read("APT_BASE.csv"), read("APT_RWY.csv"), read("APT_RWY_END.csv")
print("NASR effective date:", base["EFF_DATE"].unique())
print(f"APT_BASE {len(base):,} · APT_RWY {len(rwy):,} · APT_RWY_END {len(end):,}")

NASR effective date: <ArrowStringArray>
['2026/10/29']
Length: 1, dtype: str
APT_BASE 19,434 · APT_RWY 23,195 · APT_RWY_END 39,835


## Airports
Keep only `SITE_TYPE_CODE == 'A'` (airports; this drops heliports, seaplane bases and so on). The IATA code equals the FAA ID except for the lookup table in config.

In [3]:
apt = base.loc[base["SITE_TYPE_CODE"] == "A",
               ["SITE_NO", "ARPT_ID", "ICAO_ID", "ARPT_NAME", "CITY", "STATE_CODE",
                "LAT_DECIMAL", "LONG_DECIMAL", "ELEV", "MAG_VARN", "MAG_HEMIS"]].copy()
assert not apt["ARPT_ID"].duplicated().any()
apt = apt.rename(columns={"ARPT_ID": "faa_id", "ICAO_ID": "icao_id", "LAT_DECIMAL": "lat",
                          "LONG_DECIMAL": "lon", "ELEV": "elev_ft"})
apt["iata_id"] = apt["faa_id"].map(lambda c: FAA_TO_IATA.get(c, c))
# Magnetic variation as a signed number: east positive, west negative. True heading = magnetic + variation.
apt["mag_var_deg"] = np.where(apt["MAG_HEMIS"] == "W", -apt["MAG_VARN"], apt["MAG_VARN"])

tf = TimezoneFinder()
apt["tz"] = [tf.timezone_at(lng=lo, lat=la) for la, lo in zip(apt["lat"], apt["lon"])]
print("airports without a timezone:", apt["tz"].isna().sum())
apt[apt["faa_id"].isin(["EWR", "ORD", "DJT"])]

airports without a timezone: 0


,SITE_NO,faa_id,icao_id,ARPT_NAME,CITY,STATE_CODE,lat,lon,elev_ft,MAG_VARN,MAG_HEMIS,iata_id,mag_var_deg,tz
3352,3555.0,DJT,KDJT,PRESIDENT DONALD J TRUMP INTL,WEST PALM BEACH,FL,26.683162,-80.095592,19.6,6.0,W,PBI,-6.0,America/New_York
4287,4508.0,ORD,KORD,CHICAGO O'HARE INTL,CHICAGO,IL,41.976940,-87.908150,680.0,3.0,W,ORD,-3.0,America/Chicago
10074,14052.0,EWR,KEWR,NEWARK LIBERTY INTL,NEWARK,NJ,40.692481,-74.168688,17.5,13.0,W,EWR,-13.0,America/New_York


## Runways and runway ends
Filters:
- no helipads (`RWY_ID` starting with H)
- `RWY_LEN` ≥ 5,000 ft
- asphalt or concrete surface

Heading: `TRUE_ALIGNMENT` (degrees true, the same reference GHCNh uses for wind direction). If it's blank, use:
1. the bearing from this end's threshold to the opposite end's threshold, or
2. runway number × 10 + magnetic variation.

In [4]:
r = rwy[(rwy["SITE_TYPE_CODE"] == "A")
        & ~rwy["RWY_ID"].str.upper().str.startswith("H")
        & (rwy["RWY_LEN"] >= RUNWAY_MIN_LEN_FT)
        & rwy["SURFACE_TYPE_CODE"].str.contains("|".join(PAVED_SURFACES), na=False)]
e = end.merge(r[["SITE_NO", "RWY_ID", "RWY_LEN", "RWY_WIDTH", "SURFACE_TYPE_CODE"]],
              on=["SITE_NO", "RWY_ID"], how="inner")
e = e.merge(apt[["SITE_NO", "faa_id", "mag_var_deg"]], on="SITE_NO", how="inner")
print(f"{len(r):,} runways, {len(e):,} runway ends pass the filters")


def bearing(lat1, lon1, lat2, lon2):
    """Initial great-circle bearing in degrees true."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dl = np.radians(lon2 - lon1)
    x = np.sin(dl) * np.cos(p2)
    y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


# The opposite end of the same physical runway
opp = e[["SITE_NO", "RWY_ID", "RWY_END_ID", "LAT_DECIMAL", "LONG_DECIMAL"]].rename(
    columns={"RWY_END_ID": "OPP_END_ID", "LAT_DECIMAL": "OPP_LAT", "LONG_DECIMAL": "OPP_LON"})
e = e.merge(opp, on=["SITE_NO", "RWY_ID"], how="left")
e = e[(e["OPP_END_ID"] != e["RWY_END_ID"]) | e["OPP_END_ID"].isna()]
e = e.drop_duplicates(["SITE_NO", "RWY_ID", "RWY_END_ID"])

hdg_coords = bearing(e["LAT_DECIMAL"], e["LONG_DECIMAL"], e["OPP_LAT"], e["OPP_LON"])
rwy_num = pd.to_numeric(e["RWY_END_ID"].str.extract(r"^(\d{1,2})")[0], errors="coerce")
hdg_number = (rwy_num * 10 + e["mag_var_deg"]) % 360

e["true_hdg"] = e["TRUE_ALIGNMENT"]
e["hdg_source"] = np.where(e["true_hdg"].notna(), "TRUE_ALIGNMENT", None)
fill1 = e["true_hdg"].isna() & hdg_coords.notna()
e.loc[fill1, "true_hdg"] = hdg_coords[fill1]
e.loc[fill1, "hdg_source"] = "end_coordinates"
fill2 = e["true_hdg"].isna() & hdg_number.notna()
e.loc[fill2, "true_hdg"] = hdg_number[fill2]
e.loc[fill2, "hdg_source"] = "runway_number+magvar"
print(e["hdg_source"].value_counts(dropna=False))

2,711 runways, 5,422 runway ends pass the filters
hdg_source
TRUE_ALIGNMENT          4980
end_coordinates          242
runway_number+magvar     168
NaN                       32
Name: count, dtype: int64


## ILS flag
**ILS (Instrument Landing System)** guides aircraft down when pilots can't see the runway. In low ceilings or visibility, arrivals can in practice only use ILS-equipped ends. That's why 05 counts usable *ILS* ends separately. `LOC/GS` is a localizer plus glideslope, i.e. a full ILS. Localizer-only, LDA and SDF approaches are non-precision, so they don't count.

In [5]:
print(e["ILS_TYPE"].value_counts(dropna=False))
e["has_ils"] = e["ILS_TYPE"].isin(ILS_TYPES)

runway_ends = (e.rename(columns={"RWY_ID": "rwy_id", "RWY_END_ID": "end_id", "RWY_LEN": "len_ft",
                                 "RWY_WIDTH": "width_ft", "SURFACE_TYPE_CODE": "surface"})
               .assign(physical_runway=lambda d: d["faa_id"] + ":" + d["rwy_id"])
               [["faa_id", "rwy_id", "end_id", "physical_runway", "true_hdg", "hdg_source",
                 "len_ft", "width_ft", "surface", "has_ils"]]
               .dropna(subset=["true_hdg"])
               .sort_values(["faa_id", "rwy_id", "end_id"])
               .reset_index(drop=True))

ILS_TYPE
NaN          3970
ILS/DME       773
ILS           466
LOC/DME        88
LOC/GS         76
LOCALIZER      30
LDA/DME        18
SDF             1
Name: count, dtype: int64


## Checks

In [6]:
ewr = runway_ends[runway_ends["faa_id"] == "EWR"]
display(ewr)
assert len(ewr) == 6, "EWR should have 6 usable ends (04L/22R, 04R/22L, 11/29)"

# Opposite ends should differ by about 180°
pairs = runway_ends.merge(runway_ends, on="physical_runway", suffixes=("", "_o"))
pairs = pairs[pairs["end_id"] < pairs["end_id_o"]]
diff = ((pairs["true_hdg"] - pairs["true_hdg_o"]).abs() - 180).abs()
print(f"opposite-end heading check: {(diff > 3).sum()} of {len(pairs):,} pairs off by > 3°")
display(pairs.loc[diff > 3, ["physical_runway", "end_id", "true_hdg", "end_id_o", "true_hdg_o"]].head(10))

,faa_id,rwy_id,end_id,physical_runway,true_hdg,hdg_source,len_ft,width_ft,surface,has_ils
1932,EWR,04L/22R,04L,EWR:04L/22R,26.0,TRUE_ALIGNMENT,11000,150,ASPH-CONC,True
1933,EWR,04L/22R,22R,EWR:04L/22R,206.0,TRUE_ALIGNMENT,11000,150,ASPH-CONC,True
1934,EWR,04R/22L,04R,EWR:04R/22L,26.0,TRUE_ALIGNMENT,9999,150,ASPH,True
1935,EWR,04R/22L,22L,EWR:04R/22L,206.0,TRUE_ALIGNMENT,9999,150,ASPH,True
1936,EWR,11/29,11,EWR:11/29,95.0,TRUE_ALIGNMENT,6725,150,ASPH,True
1937,EWR,11/29,29,EWR:11/29,275.0,TRUE_ALIGNMENT,6725,150,ASPH,False


opposite-end heading check: 3 of 2,695 pairs off by > 3°


,physical_runway,end_id,true_hdg,end_id_o,true_hdg_o
2793,CYQH:08/26,08,8.0,26,265.0
2953,CYYZ:15L/33R,15L,147.0,33R,237.0
2957,CYYZ:15R/33L,15R,147.0,33L,237.0


Every airport code in the BTS files has to resolve to a NASR airport with a timezone.

In [7]:
bts_files = sorted(BTS_DIR.glob("*.parquet"))
if bts_files:
    codes = set()
    for p in bts_files:
        od = pd.read_parquet(p, columns=["ORIGIN", "DEST"])
        codes |= set(od["ORIGIN"].dropna()) | set(od["DEST"].dropna())
else:
    od = pd.read_csv(RAW / "bts_manual" / "T_ONTIME_REPORTING-2.csv", usecols=["ORIGIN", "DEST"])
    codes = set(od["ORIGIN"]) | set(od["DEST"])
known = set(apt.loc[apt["tz"].notna(), "faa_id"])
unresolved = sorted(c for c in codes if to_faa(c) not in known)
print(f"{len(codes)} BTS airport codes across {max(len(bts_files), 1)} file(s); unresolved: {unresolved}")
assert not unresolved, "add these to IATA_TO_FAA in config"

370 BTS airport codes across 45 file(s); unresolved: []


In [8]:
airports = apt[["faa_id", "iata_id", "icao_id", "ARPT_NAME", "CITY", "STATE_CODE",
                "lat", "lon", "elev_ft", "mag_var_deg", "tz"]].rename(
    columns={"ARPT_NAME": "name", "CITY": "city", "STATE_CODE": "state"})
runway_ends.to_parquet(INTERIM / "runway_ends.parquet", index=False)
airports.to_parquet(INTERIM / "airports.parquet", index=False)
print(f"saved {len(runway_ends):,} runway ends and {len(airports):,} airports")

saved 5,390 runway ends and 13,005 airports
